# Hybrid Neural Systems in Healthcare: CNN + LSTM Implementation

**Google Colab version — beginner-friendly, end-to-end notebook**

This notebook builds a hybrid 1D-CNN + LSTM model using time-series vital signs. The CNN extracts local temporal patterns; the LSTM learns sequence dependencies; a dense layer produces a simulated binary risk-pattern prediction.

> **Important:** The dataset and labels generated here are synthetic and illustrative. This is an educational software demonstration, not a clinically validated model. Do not use it for diagnosis, triage, treatment, or real patient decisions.


## 1. Environment setup

Colab usually includes PyTorch, NumPy, scikit-learn, and Matplotlib. Run this cell to check the installed versions. If an import fails, uncomment the pip install line and run it.


In [ ]:
# Optional: uncomment if a package is missing
# %pip install -q numpy scikit-learn matplotlib torch

import sys, random, json, zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")


## 2. Generate a synthetic demonstration dataset

Each patient is represented by 24 time steps and five features:
1. Heart rate
2. Systolic blood pressure
3. Oxygen saturation (SpO2)
4. Respiratory rate
5. Temperature

The labels are assigned using a simple illustrative rule, not a medical diagnosis.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

N_PATIENTS = 1200
TIME_STEPS = 24
N_FEATURES = 5

centers = np.array([78.0, 120.0, 97.0, 16.0, 36.8])
scales = np.array([8.0, 10.0, 1.0, 2.0, 0.25])

X = np.random.normal(
    centers, scales, size=(N_PATIENTS, TIME_STEPS, N_FEATURES)
)

# Add patient-specific smooth trends.
trend = np.random.normal(0, 1, size=(N_PATIENTS, 1, N_FEATURES))
time_axis = np.linspace(-0.5, 0.5, TIME_STEPS)[None, :, None]
X += trend * time_axis * np.array([5.0, 6.0, 0.8, 1.5, 0.15])[None, None, :]

# Add a subset with progressively abnormal-looking synthetic patterns.
latent_pattern = np.random.random(N_PATIENTS) < 0.38
drift = np.linspace(0, 1, TIME_STEPS)
for i in np.where(latent_pattern)[0]:
    X[i, :, 0] += 12 * drift
    X[i, :, 1] -= 13 * drift
    X[i, :, 2] -= 3.0 * drift
    X[i, :, 3] += 5 * drift
    X[i, :, 4] += 0.6 * drift

# Transparent illustrative label rule.
score = (
    (X[:, :, 0].mean(axis=1) > 90).astype(int)
    + (X[:, :, 1].min(axis=1) < 105).astype(int)
    + (X[:, :, 2].min(axis=1) < 94).astype(int)
    + (X[:, :, 3].max(axis=1) > 20).astype(int)
    + (X[:, :, 4].max(axis=1) > 37.5).astype(int)
)
y = (score >= 2).astype(np.int64)

X = X.astype(np.float32)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class counts:", np.bincount(y))
print("Positive-label proportion:", round(float(y.mean()), 3))


## 3. Split the data and normalize features

We split by patient/sample. The scaler is fitted only on the training set to reduce data leakage.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, N_FEATURES))

def scale_sequences(arr):
    original_shape = arr.shape
    scaled = scaler.transform(arr.reshape(-1, N_FEATURES))
    return scaled.reshape(original_shape).astype(np.float32)

X_train = scale_sequences(X_train)
X_val = scale_sequences(X_val)
X_test = scale_sequences(X_test)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)


## 4. Define the hybrid CNN + LSTM architecture

- **1D CNN:** detects local patterns across adjacent time steps.
- **LSTM:** learns how the extracted patterns change over time.
- **Dense classifier:** outputs one logit for binary classification.


In [ ]:
class CNNLSTM(nn.Module):
    def __init__(self, n_features=5, hidden_size=32):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(n_features, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        # Input: (batch, time_steps, features)
        x = x.transpose(1, 2)   # (batch, features, time_steps)
        x = self.cnn(x)         # (batch, 64, reduced_time)
        x = x.transpose(1, 2)   # (batch, reduced_time, 64)
        _, (hidden, _) = self.lstm(x)
        return self.classifier(hidden[-1]).squeeze(1)  # logits

model = CNNLSTM(n_features=N_FEATURES)
print(model)


## 5. Train the model

The notebook uses binary cross-entropy with logits, Adam optimization, validation loss, and early stopping.


In [ ]:
def make_loader(features, labels, shuffle=False, batch_size=32):
    dataset = TensorDataset(
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNLSTM(n_features=N_FEATURES).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 25
PATIENCE = 5
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_total += loss.item() * len(xb)

    train_loss = train_total / len(train_loader.dataset)

    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_total += criterion(model(xb), yb).item() * len(xb)
    val_loss = val_total / len(val_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)


## 6. Evaluate on the held-out test set

We report accuracy, precision, recall/sensitivity, F1-score, ROC-AUC (when possible), and a confusion matrix. Because the data are synthetic, these values only demonstrate that the code runs; they are not clinical performance results.


In [ ]:
model.eval()
test_probabilities = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        probs = torch.sigmoid(model(xb)).cpu().numpy()
        test_probabilities.extend(probs.tolist())

test_probabilities = np.asarray(test_probabilities)
test_predictions = (test_probabilities >= 0.5).astype(int)

metrics = {
    "accuracy": float(accuracy_score(y_test, test_predictions)),
    "precision": float(precision_score(y_test, test_predictions, zero_division=0)),
    "recall_sensitivity": float(recall_score(y_test, test_predictions, zero_division=0)),
    "f1_score": float(f1_score(y_test, test_predictions, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, test_probabilities))
        if len(np.unique(y_test)) == 2 else None,
    "test_samples": int(len(y_test)),
    "threshold": 0.5,
    "warning": "Synthetic educational data only; not clinical performance."
}

for name, value in metrics.items():
    print(f"{name}: {value}")

cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Lower-risk pattern", "Higher-risk pattern"]
).plot()
plt.title("Test confusion matrix — synthetic data")
plt.tight_layout()
plt.show()


## 7. Plot training and validation loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("CNN–LSTM training curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 8. Make one illustrative prediction

This creates a fresh synthetic vital-sign sequence, scales it with the training scaler, and passes it through the trained model. It is only a code demonstration—not an assessment of a real person.


In [ ]:
rng = np.random.default_rng(7)
example_sequence = np.column_stack([
    rng.normal(92, 5, TIME_STEPS),       # heart rate
    rng.normal(112, 8, TIME_STEPS),      # systolic BP
    rng.normal(95, 1.5, TIME_STEPS),     # SpO2
    rng.normal(19, 2, TIME_STEPS),       # respiratory rate
    rng.normal(37.2, 0.3, TIME_STEPS),   # temperature
]).astype(np.float32)

example_scaled = scaler.transform(example_sequence).astype(np.float32)
model.eval()
with torch.no_grad():
    input_tensor = torch.tensor(example_scaled[None, :, :]).to(device)
    probability = torch.sigmoid(model(input_tensor)).item()

print(f"Illustrative model score: {probability:.3f}")
print("Predicted pattern:", "Higher-risk pattern" if probability >= 0.5 else "Lower-risk pattern")
print("Educational demonstration only — not a medical assessment.")


## 9. Save the model, metrics, and plots

This creates a `colab_outputs` folder and a ZIP archive you can download and add to your GitHub repository. The synthetic raw dataset is not included in the ZIP.


In [ ]:
OUTPUT_DIR = Path("colab_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

torch.save(model.state_dict(), OUTPUT_DIR / "cnn_lstm_model.pt")
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(OUTPUT_DIR / "scaler.json").write_text(json.dumps({
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "feature_order": [
        "heart_rate", "systolic_bp", "spo2",
        "respiratory_rate", "temperature"
    ]
}, indent=2))

# Save training curve.
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("CNN–LSTM training curve")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curve.png", dpi=150)
plt.close()

# Save confusion matrix.
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Lower-risk", "Higher-risk"]
).plot(ax=ax)
ax.set_title("Test confusion matrix (synthetic data)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.close(fig)

zip_name = "CNN_LSTM_Colab_Results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in OUTPUT_DIR.iterdir():
        zf.write(file, arcname=file.name)

print("Saved outputs:", [p.name for p in OUTPUT_DIR.iterdir()])
print("Created:", zip_name)


## 10. Download results from Colab

Run this cell to download the results ZIP to your computer. You can upload it to the `Implementation/results/` folder in your GitHub repository. Do not upload private patient data.


In [ ]:
from google.colab import files
files.download("CNN_LSTM_Colab_Results.zip")


## 11. Dataset upgrade for future research

For a research-grade study, replace the synthetic data with a properly licensed and de-identified healthcare time-series dataset. Check its access terms, ethics requirements, and citation instructions. Prepare arrays with:
- `X`: `(number_of_patients, time_steps, 5)`
- `y`: `(number_of_patients,)`, binary labels 0/1
- Feature order: heart rate, systolic BP, SpO2, respiratory rate, temperature

Keep all records from one patient in only one split. Do not randomly split individual time windows from the same patient across train and test sets. Never commit identifiable or confidential patient information to GitHub.

## 12. Suggested GitHub structure

```text
Hybrid-Neural-Systems-Healthcare/
├── README.md
├── Research-Paper/
│   ├── Research_Paper.pdf
│   └── Research_Paper.docx
└── Implementation/
    ├── Hybrid_Neural_Systems_CNN_LSTM_Colab.ipynb
    ├── README.md
    ├── requirements.txt
    └── results/
        ├── cnn_lstm_model.pt
        ├── metrics.json
        ├── scaler.json
        ├── training_curve.png
        └── confusion_matrix.png
```
